In [ ]:
# ===============================
# STEP 1: INSTALL REQUIRED LIBRARIES
# ===============================

# For data handling
!pip install -q pandas numpy

# For machine learning
!pip install -q scikit-learn

# For text preprocessing (stopwords, stemming, etc.)
!pip install -q nltk

# For saving/loading trained model
!pip install -q joblib

# For visualization (optional, but useful for word clouds and plots)
!pip install -q matplotlib wordcloud


In [ ]:
# ===============================
# STEP 2: IMPORT LIBRARIES
# ===============================
import re, string, joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# Download stopwords (first run only)
nltk.download("stopwords")


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
# ===============================
# STEP 3: LOAD DATASET FROM GOOGLE DRIVE
# ===============================

import pandas as pd

# Give the full path of your CSV file
file_path = "/content/drive/MyDrive/sem III/Phishing_validation_emails (2).csv"

# Load dataset
df_raw = pd.read_csv(file_path)

print("\n✅ Dataset loaded successfully!")
print("Shape:", df_raw.shape)
print("Columns:", df_raw.columns.tolist())
df_raw.head()



✅ Dataset loaded successfully!
Shape: (2000, 2)
Columns: ['Email Text', 'Email Type']


,Email Text,Email Type
0,"Dear Jordan, your subscription has been succes...",Safe Email
1,"Dear Casey, thank you for your purchase. Your ...",Safe Email
2,Congratulations! You've won a $3000 gift card....,Phishing Email
3,You have a new secure message from your bank. ...,Phishing Email
4,Your package delivery is pending. Please provi...,Phishing Email


In [ ]:
# ===============================
# STEP 4: AUTO-DETECT COLUMNS & CLEAN LABELS
# ===============================

# Convert column names to lowercase for easy matching
cols_lower = [c.lower() for c in df_raw.columns]

# Candidate names for text & label columns
text_candidates  = ["text", "email text", "message", "content", "body"]
label_candidates = ["label", "class", "target", "category", "email type", "is_spam", "spam_label"]

def find_match(candidates):
    for cand in candidates:
        if cand in cols_lower:
            return df_raw.columns[cols_lower.index(cand)]
    return None

text_col  = find_match(text_candidates)
label_col = find_match(label_candidates)

if text_col is None or label_col is None:
    raise ValueError("❌ Could not auto-detect text/label columns. Found: " + str(df_raw.columns))

print(f"\n✅ Using '{text_col}' as TEXT column and '{label_col}' as LABEL column")

# Keep only these columns and rename
df = df_raw[[text_col, label_col]].rename(columns={text_col: "text", label_col: "label"}).copy()
df["text"] = df["text"].astype(str).fillna("")

# Normalize labels
def normalize_label(x):
    x = str(x).strip().lower()
    if x in {"spam","phishing","malicious","fraud"}:
        return 1
    if x in {"ham","legit","legitimate","not_spam","nonspam","normal"}:
        return 0
    if x.isdigit():
        return 1 if int(x) == 1 else 0
    return 0  # default: legit

df["label"] = df["label"].apply(normalize_label)

print("\n📊 Label distribution:")
print(df["label"].value_counts())
df.head()



✅ Using 'Email Text' as TEXT column and 'Email Type' as LABEL column

📊 Label distribution:
label
0    2000
Name: count, dtype: int64


,text,label
0,"Dear Jordan, your subscription has been succes...",0
1,"Dear Casey, thank you for your purchase. Your ...",0
2,Congratulations! You've won a $3000 gift card....,0
3,You have a new secure message from your bank. ...,0
4,Your package delivery is pending. Please provi...,0


In [ ]:
# ===============================
# STEP 5: TEXT PREPROCESSING
# ===============================

import re, string
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

STOPWORDS = set(stopwords.words("english"))
STEMMER = PorterStemmer()

# Regex patterns
URL_RE    = re.compile(r"https?://\S+|www\.\S+")
EMAIL_RE  = re.compile(r"\b[\w\.-]+@[\w\.-]+\.\w+\b")
HTML_RE   = re.compile(r"<.*?>")
NUM_RE    = re.compile(r"\b\d+(?:\.\d+)?\b")
PUNCT_TBL = str.maketrans("", "", string.punctuation)

def clean_text(s: str) -> str:
    s = s.lower()
    s = HTML_RE.sub(" ", s)
    s = URL_RE.sub(" URL ", s)
    s = EMAIL_RE.sub(" EMAIL ", s)
    s = NUM_RE.sub(" NUM ", s)
    s = s.translate(PUNCT_TBL)
    tokens = s.split()
    tokens = [STEMMER.stem(t) for t in tokens if t not in STOPWORDS]
    return " ".join(tokens)

# Apply cleaning
df["clean_text"] = df["text"].apply(clean_text)

print("\n✅ Preprocessing done! Sample cleaned emails:")
df[["text", "clean_text", "label"]].head(10)



✅ Preprocessing done! Sample cleaned emails:


,text,clean_text,label
0,"Dear Jordan, your subscription has been succes...",dear jordan subscript success renew thank cont...,0
1,"Dear Casey, thank you for your purchase. Your ...",dear casey thank purchas order ship soon,0
2,Congratulations! You've won a $3000 gift card....,congratul youv num gift card click claim prize,0
3,You have a new secure message from your bank. ...,new secur messag bank click read,0
4,Your package delivery is pending. Please provi...,packag deliveri pend pleas provid person infor...,0
5,"Hi Drew, it was great meeting you at the confe...",hi drew great meet confer let catch coffe next...,0
6,Your package delivery is pending. Please provi...,packag deliveri pend pleas provid person infor...,0
7,"Hi Alex, it was great meeting you at the confe...",hi alex great meet confer let catch coffe next...,0
8,Alert: Unusual login attempt detected. Verify ...,alert unusu login attempt detect verifi accoun...,0
9,Your subscription is about to expire. Renew no...,subscript expir renew continu enjoy servic,0


In [ ]:
# ===============================
# STEP 6: TRAIN / TEST SPLIT
# ===============================

from sklearn.model_selection import train_test_split

X = df["clean_text"]   # input features (processed text)
y = df["label"]        # target labels (0 = legit, 1 = spam/phishing)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y   # keep same spam/ham ratio
)

print("✅ Data split completed!")
print("Training size:", len(X_train))
print("Testing size:", len(X_test))


✅ Data split completed!
Training size: 1500
Testing size: 500


In [ ]:
# ===============================
# STEP 7: TRAIN NAÏVE BAYES MODEL
# ===============================

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

# Create a pipeline: TF-IDF + Naive Bayes
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1,2),    # use unigrams + bigrams
        min_df=2,             # ignore very rare words
        max_df=0.98,          # ignore overly common words
        sublinear_tf=True     # log-scale term frequency
    )),
    ("nb", MultinomialNB(alpha=1.0))  # Laplace smoothing
])

# Train the model
pipeline.fit(X_train, y_train)

print("✅ Model training completed!")


✅ Model training completed!


In [ ]:
# ===============================
# STEP 8: EVALUATE THE MODEL
# ===============================

from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# Predict on test set
y_pred = pipeline.predict(X_test)

# Accuracy
print("✅ Accuracy:", accuracy_score(y_test, y_pred))

# Detailed classification report
print("\n📄 Classification Report:\n")
print(classification_report(y_test, y_pred, digits=4))

# Confusion matrix
print("\n🧮 Confusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))


✅ Accuracy: 1.0

📄 Classification Report:

              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000       500

    accuracy                         1.0000       500
   macro avg     1.0000    1.0000    1.0000       500
weighted avg     1.0000    1.0000    1.0000       500


🧮 Confusion Matrix:

[[500]]


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(


In [ ]:
# ===============================
# STEP 9: SAVE THE TRAINED MODEL
# ===============================

import joblib

# Save model to a file
joblib.dump(pipeline, "naive_bayes_spam_model.joblib")

print("💾 Model saved as 'naive_bayes_spam_model.joblib'")


💾 Model saved as 'naive_bayes_spam_model.joblib'


In [ ]:
# Load the saved model
loaded_model = joblib.load("naive_bayes_spam_model.joblib")

# Quick test
sample = "Congratulations! You have won a lottery. Click the link."
print("Prediction:", loaded_model.predict([sample])[0])


Prediction: 0


In [ ]:
# ===============================
# FIXED PREDICTION FUNCTION
# ===============================

def predict_sentence(s: str):
    cleaned = clean_text(s)

    # Predict probabilities (handle both 1-class and 2-class cases)
    probs = loaded_model.predict_proba([cleaned])[0]

    # If model has only one class in training
    if probs.shape[0] == 1:
        prob_legit = 1.0 if loaded_model.classes_[0] == 0 else 0.0
        prob_spam  = 1.0 if loaded_model.classes_[0] == 1 else 0.0
    else:
        # normal case: two probabilities [legit, spam]
        prob_legit, prob_spam = probs[0], probs[1]

    # ML predicted label
    ml_label = 1 if prob_spam >= 0.5 else 0

    # Extra spam keyword check
    words = set(cleaned.split())
    if any(w in words for w in SPAM_KEYWORDS):
        final_label = 1
    else:
        final_label = ml_label

    label_str = "SPAM / PHISHING (Harmful)" if final_label == 1 else "LEGIT / HAM (Safe)"
    return {
        "input": s,
        "prediction": label_str,
        "prob_legit": float(prob_legit),
        "prob_spam": float(prob_spam)
    }

# 🔎 Test examples
print(predict_sentence("Congratulations! You have won a lottery. Click here."))
print(predict_sentence("Dear team, please find attached the project report."))


{'input': 'Congratulations! You have won a lottery. Click here.', 'prediction': 'SPAM / PHISHING (Harmful)', 'prob_legit': 1.0, 'prob_spam': 0.0}
{'input': 'Dear team, please find attached the project report.', 'prediction': 'LEGIT / HAM (Safe)', 'prob_legit': 1.0, 'prob_spam': 0.0}
